In [1]:
import torch
print(torch.cuda.is_available())
!nvidia-smi

True
Mon Mar 30 17:08:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+

In [2]:
!pip install -q transformers accelerate peft bitsandbytes trl datasets safetensors einops

In [3]:
import os
from huggingface_hub import login

login(os.getenv("HF_TOKEN"))

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [5]:
model_name = "microsoft/Phi-4-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

In [6]:
for name, module in model.named_modules():
    if "proj" in name.lower():
        print(name)

model.layers.0.self_attn.o_proj
model.layers.0.self_attn.qkv_proj
model.layers.0.mlp.gate_up_proj
model.layers.0.mlp.down_proj
model.layers.1.self_attn.o_proj
model.layers.1.self_attn.qkv_proj
model.layers.1.mlp.gate_up_proj
model.layers.1.mlp.down_proj
model.layers.2.self_attn.o_proj
model.layers.2.self_attn.qkv_proj
model.layers.2.mlp.gate_up_proj
model.layers.2.mlp.down_proj
model.layers.3.self_attn.o_proj
model.layers.3.self_attn.qkv_proj
model.layers.3.mlp.gate_up_proj
model.layers.3.mlp.down_proj
model.layers.4.self_attn.o_proj
model.layers.4.self_attn.qkv_proj
model.layers.4.mlp.gate_up_proj
model.layers.4.mlp.down_proj
model.layers.5.self_attn.o_proj
model.layers.5.self_attn.qkv_proj
model.layers.5.mlp.gate_up_proj
model.layers.5.mlp.down_proj
model.layers.6.self_attn.o_proj
model.layers.6.self_attn.qkv_proj
model.layers.6.mlp.gate_up_proj
model.layers.6.mlp.down_proj
model.layers.7.self_attn.o_proj
model.layers.7.self_attn.qkv_proj
model.layers.7.mlp.gate_up_proj
model.layers.

In [7]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules = ["qkv_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 3,670,016 || all params: 3,839,691,776 || trainable%: 0.0956


In [8]:
dataset = load_dataset("tatsu-lab/alpaca")
dataset.shuffle(seed=42)
dataset["train"] = dataset["train"].select(range(5000))

In [9]:
def format_prompt(example):
    return {
        "text": f"""<|user|>
{example['instruction']}
<|assistant|>
{example['output']}"""
    }

dataset = dataset.map(format_prompt)

def formatting_func(example):
    return example["text"]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [10]:
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,

    fp16=False,
    bf16=False,

    logging_steps=5,
    output_dir="./results",
    report_to="none"
)

In [11]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    formatting_func=formatting_func,
    processing_class=tokenizer,
    args=training_args
)

Applying formatting function to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [12]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,3.732360
10,3.524596
15,3.224254
20,2.414002
25,2.604218
30,2.301268
35,2.491035
40,2.089826
45,2.038841
50,1.569958


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=5000, training_loss=1.6099232039451599, metrics={'train_runtime': 2593.2992, 'train_samples_per_second': 1.928, 'train_steps_per_second': 1.928, 'total_flos': 6380192870473728.0, 'train_loss': 1.6099232039451599})

In [19]:
'''
from huggingface_hub import login
import os

login(os.getenv("HF_TOKEN"))


repo_name = "your-username/phi-mini-finetuned"   # 🔥 change username


trainer.model.push_to_hub(repo_name)


tokenizer.push_to_hub(repo_name)
'''

'\nfrom huggingface_hub import login\nimport os\n\nlogin(os.getenv("HF_TOKEN"))\n\n\nrepo_name = "your-username/phi-mini-finetuned"   # 🔥 change username\n\n\ntrainer.model.push_to_hub(repo_name)\n\n\ntokenizer.push_to_hub(repo_name)\n'